# M6 v2 — Contrôle qualité & complétion de la réconciliation

**Orange Money | Koceila SALEM**

## Repositionnement honnête (vs v1)
La v1 recopiait les liens déjà établis (jointure SQL, peu de valeur). La v2 apporte
une vraie valeur RA&FM en 3 axes :

1. **AUDIT** : détecter les réconciliations existantes SUSPECTES (montant/délai incohérents)
2. **COMPLÉTION** : apparier les corrections/rollbacks ORPHELINS (sans lien)
3. **VALIDATION HONNÊTE** : mesurer les faux positifs sur la vérité terrain

## Méthode de validation
On cache volontairement 20% des liens connus, on tente de les retrouver,
et on mesure précision/rappel RÉELS. Pas de confiance circulaire.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np, json, time, warnings
warnings.filterwarnings('ignore')
from src import config as cfg
from src.data_loader import load_parquet

MODEL_DIR  = cfg.MODELS_DIR / 'M6_reconciliation'
OUTPUT_DIR = cfg.OUTPUTS_DIR / 'M6_reconciliation'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OK')

In [ ]:
COLS = [cfg.COL_TRANSFER_ID, cfg.COL_DATE, cfg.COL_MONTANT, cfg.COL_SERVICE,
        cfg.COL_SUBTYPE, cfg.COL_STATUT, cfg.COL_ACTION, cfg.COL_TAG,
        cfg.COL_RECON_BY, cfg.COL_RECON_FOR, cfg.COL_SENDER_ID, cfg.COL_RECVR_ID]
COLS = list(dict.fromkeys(COLS))
df = load_parquet(columns=COLS)
df[cfg.COL_MONTANT] = pd.to_numeric(df[cfg.COL_MONTANT], errors='coerce').fillna(0)
df_u = df.drop_duplicates(subset=[cfg.COL_TRANSFER_ID], keep='first')
print(f'{len(df):,} transactions | {len(df_u):,} TRANSFER_ID uniques')

## AXE 1 — AUDIT des réconciliations existantes

Détecter les liens DÉJÀ établis mais SUSPECTS (le système source peut se tromper).

In [ ]:
idx_montant = df_u.set_index(cfg.COL_TRANSFER_ID)[cfg.COL_MONTANT]
idx_date    = df_u.set_index(cfg.COL_TRANSFER_ID)[cfg.COL_DATE]

rec = df[df[cfg.COL_RECON_FOR].notna()].copy()
rec['j_montant'] = rec[cfg.COL_RECON_FOR].map(idx_montant)
rec['j_date']    = rec[cfg.COL_RECON_FOR].map(idx_date)
rec['lien_valide'] = rec['j_montant'].notna()

audit = rec[rec['lien_valide']].copy()
audit['ecart_montant'] = np.abs(audit[cfg.COL_MONTANT] - audit['j_montant'])
audit['delai_h'] = (audit[cfg.COL_DATE]-audit['j_date']).abs().dt.total_seconds()/3600

# Critères d'audit (anomalies dans les liens existants)
audit['anomalie_montant'] = (audit['ecart_montant'] > 1).astype(int)
audit['anomalie_delai']   = (audit['delai_h'] > 168).astype(int)  # > 7 jours
audit['lien_brise']       = (~rec['lien_valide']).astype(int) if False else 0
audit['SUSPECT'] = ((audit['anomalie_montant']==1) | (audit['anomalie_delai']==1)).astype(int)

n_suspect = audit['SUSPECT'].sum()
lien_brise = (~rec['lien_valide']).sum()
print('=== AUDIT DES RÉCONCILIATIONS EXISTANTES ===')
print(f'Liens audités        : {len(audit):,}')
print(f'Liens BRISÉS (cible inexistante) : {lien_brise:,}')
print(f'Montant incohérent   : {audit["anomalie_montant"].sum():,}')
print(f'Délai > 7j           : {audit["anomalie_delai"].sum():,}')
print(f'TOTAL SUSPECTS       : {n_suspect:,} ({n_suspect/len(audit)*100:.2f}%)')

if n_suspect > 0:
    print('\nExemples de liens suspects :')
    print(audit[audit['SUSPECT']==1][[cfg.COL_TRANSFER_ID, cfg.COL_MONTANT,
          'j_montant','ecart_montant','delai_h']].head(10).to_string(index=False))

## AXE 2 — VALIDATION HONNÊTE par vérité terrain cachée

On cache 20% des liens connus, on tente de les retrouver par règles,
et on mesure la précision/rappel RÉELS (pas circulaire).

In [ ]:
# Paires connues fiables (lien valide + montant cohérent) = vérité terrain
paires = audit[(audit['lien_valide']) & (audit['ecart_montant']<1)].copy()
print(f'Paires fiables (vérité terrain) : {len(paires):,}')

test = paires.sample(frac=0.2, random_state=cfg.RANDOM_SEED)
test_sample = test.head(5000)
print(f'Paires de test (lien caché) : {len(test_sample):,}')

# ── OPTIMISATION searchsorted : chaque montant -> arrays triés par temps ──
print('Pré-indexation par montant (arrays triés)...')
t0 = time.time()
df_u2 = df_u[[cfg.COL_TRANSFER_ID, cfg.COL_MONTANT, cfg.COL_DATE]].dropna().copy()
df_u2['_m'] = df_u2[cfg.COL_MONTANT].round(0)
df_u2 = df_u2.sort_values('_m')

# Pour chaque montant : (ids triés par temps, temps_ns triés)
index_montant = {}
for m, g in df_u2.groupby('_m', sort=False):
    gg = g.sort_values(cfg.COL_DATE)
    ids = gg[cfg.COL_TRANSFER_ID].to_numpy()
    temps = gg[cfg.COL_DATE].to_numpy(dtype='datetime64[ns]').astype('int64')
    index_montant[m] = (ids, temps)
print(f'  {len(index_montant):,} montants indexés | {time.time()-t0:.0f}s')

# ── Recherche par searchsorted (logarithmique) ──
t0 = time.time()
H24 = 24*3600*1_000_000_000  # 24h en ns
vrais_positifs = faux_positifs = non_retrouve = 0

for _, row in test_sample.iterrows():
    m = round(row[cfg.COL_MONTANT])
    t = np.datetime64(row[cfg.COL_DATE], 'ns').astype('int64')
    vrai_jumeau = row[cfg.COL_RECON_FOR]
    mon_id = row[cfg.COL_TRANSFER_ID]
    entry = index_montant.get(m)
    if entry is None:
        non_retrouve += 1; continue
    ids, temps = entry
    # position d'insertion = candidat le plus proche dans le temps
    pos = np.searchsorted(temps, t)
    # examiner les 2 voisins (avant/après) pour trouver le plus proche != soi
    best, best_delai = None, H24
    for p in (pos-1, pos, pos+1):
        if 0 <= p < len(ids) and ids[p] != mon_id:
            d = abs(temps[p]-t)
            if d < best_delai:
                best_delai = d; best = ids[p]
    if best is None:
        non_retrouve += 1; continue
    if best == vrai_jumeau:
        vrais_positifs += 1
    else:
        faux_positifs += 1

duree = time.time()-t0
n = len(test_sample)
precision = vrais_positifs/(vrais_positifs+faux_positifs+1e-9)
rappel = vrais_positifs/n
print(f'\n=== VALIDATION HONNÊTE (sur {n:,} cas test, {duree:.0f}s) ===')
print(f'Vrais positifs (bon jumeau)  : {vrais_positifs:,}')
print(f'Faux positifs (mauvais)      : {faux_positifs:,}')
print(f'Non retrouvés                : {non_retrouve:,}')
print(f'\nPRÉCISION : {precision*100:.1f}% (quand on apparie, on a raison)')
print(f'RAPPEL    : {rappel*100:.1f}% (part des jumeaux retrouvés)')
print(f'\n>>> VRAIES métriques, vérité terrain cachée <<<')

## AXE 3 — Interprétation : la règle montant+temps est-elle fiable ?

In [ ]:
print('=== DIAGNOSTIC DE FIABILITÉ ===\n')
if precision > 0.9:
    print(f'✅ Précision {precision*100:.0f}% : règle montant+temps FIABLE')
    print('   -> utilisable pour compléter les liens manquants en production')
elif precision > 0.7:
    print(f'⚠️  Précision {precision*100:.0f}% : fiable AVEC supervision analyste')
else:
    print(f'❌ Précision {precision*100:.0f}% : trop de faux positifs')
    print('   -> montant+temps INSUFFISANT pour l auto en production')

# Ambiguïté via l'index searchsorted
tailles = {m: len(ids) for m,(ids,_) in index_montant.items()}
ambig = [tailles.get(round(mm),0) for mm in test_sample[cfg.COL_MONTANT].head(1000)]
print(f'\nAmbiguïté : en moyenne {np.mean(ambig):.0f} transactions/montant')
print(f'Médiane : {np.median(ambig):.0f} | Max : {np.max(ambig):,}')
print('Plus c est élevé, plus l appariement par montant seul est risqué.')

## Export & synthèse

In [ ]:
# Export des liens suspects (vraie valeur RA : à vérifier par un analyste)
if n_suspect > 0:
    audit[audit['SUSPECT']==1][[cfg.COL_TRANSFER_ID, cfg.COL_RECON_FOR,
        cfg.COL_MONTANT, 'j_montant', 'ecart_montant', 'delai_h']]\
        .to_csv(OUTPUT_DIR/'liens_suspects.csv', index=False, encoding='utf-8-sig')

params = {
    'version': 'v2 - controle qualite',
    'precision_reelle': float(precision),
    'rappel_reel': float(rappel),
    'n_liens_suspects': int(n_suspect),
    'n_liens_brises': int(lien_brise),
    'ambiguite_montant_moy': float(np.mean(ambig)),
    'regle_temps_h': 24, 'regle_montant_seuil': 1,
}
with open(MODEL_DIR/'params_v2.json','w') as f:
    json.dump(params, f, indent=2, default=str)

print('=== SYNTHÈSE M6 v2 ===')
print(f'AXE 1 (audit)      : {n_suspect:,} liens suspects détectés + {lien_brise:,} liens brisés')
print(f'AXE 2 (validation) : précision {precision*100:.0f}%, rappel {rappel*100:.0f}% (vérité terrain)')
print(f'AXE 3 (fiabilité)  : {np.mean(ambig):.0f} transactions/montant (ambiguïté)')
print(f'\nVraie valeur RA : détecter les {n_suspect:,} liens douteux du système source')
print('Honnêteté : métriques mesurées sur données cachées, pas circulaires')